In [1]:
import warnings
warnings.filterwarnings('ignore')

# 벡터 저장소 검색기 활용 및 검색 성능 평가

# 라이브러리 설치

## 설치하는 라이브러리의 역할

`faiss-cpu`: 벡터 검색 엔진 라이브러리, 여러 개의 문서 조각 중에서 사용자의 질문과 의미가 가장 유사한 문서를 찾는다. 벡터스토어를 구축한다.  
`rank_bm25`: 키워드 기반 검색 알고리즘 라이브러리, 단순 키워드 일치 여부를 넘어서 단어의 희소성과 빈도를 계산해서 점수화한다.    
`kiwipiepy`: 한국어 형태소 분석기 라이브러리, 한국어 문장을 단어 단위로 쪼개고 조사를 정교하게 분리한다.  
`openpyxl`: 엑셀 파일을 읽고 쓰기 위한 라이브러리, 판다스와 함께 자주 사용된다.

In [2]:
# !pip install faiss-cpu rank_bm25 kiwipiepy openpyxl

# 환경 설정

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [4]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser

# 벡터저장소(VectorStore)

# Chroma 저장소 생성, 문서 관리, 문서 검색

## 벡터저장소 초기화

허깅 페이스 임베딩 모델을 사용해서 Chroma 벡터저장소 만든다.

In [5]:
# 다국어 처리가 뛰어난 BAAI/bge-m3 모델을 사용해서 허깅 페이스 임베딩 모델을 만든다.
# BAAI/bge-m3 모델은 한국어, 영어 등 여러 언어를 동시에 잘 처리하며, 긴 문장도 효과적으로 벡터화할 수 있어 RAG 시스템 구축시 선호되는 모델이다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

# 비어있는 Chroma 벡터저장소 만든다. 비어있는 벡터저장소를 만들 때는 from_documents() 메소드를 사용하지 않는다.
chroma_db = Chroma(
    # BAAI/bge-m3 모델을 사용해서 Chroma 벡터저장소 만들때 문자를 숫자로 바꾸는 임베딩을 한다.
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

get() 메소드는 현재 연결된 Chroma 벡터저장소에 저장된 모든 데이터를 추출하거나, 특정 조건에 맞는 데이터를 조회할 때 사용한다.

In [6]:
chroma_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

## 벡터저장소 관리

Chroma 벡터저장소에는 Document 객체를 저장해야 하므로 Document를 import 한다.

In [7]:
from langchain_core.documents import Document

In [8]:
# Chroma 벡터저장소에 저장할 원본 데이터
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]

# Document 객체를 생성한다. Document 객체에는 부가정보(metadata)와 본문(page_content)가 포함된다.
doc_objects = []
for index, document in enumerate(documents, start=1):
    # print(index, document)
    doc = Document(
        page_content = document,
        metadata = {'source': f'AI_textbook {index}', 'chapter': f'Chapter {index}'}
    )
    # print(doc)
    doc_objects.append(doc)
print(doc_objects)

# Chroma 벡터저장소에 저장되는 Document 객체의 고유 식별자(ID)를 생성한다.
doc_ids = [f'DOC_{i}' for i in range(1, len(doc_objects) + 1)]
print(doc_ids)

[Document(metadata={'source': 'AI_textbook', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'), Document(metadata={'source': 'AI_textbook', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'), Document(metadata={'source': 'AI_textbook', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'), Document(metadata={'source': 'AI_textbook', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'), Document(metadata={'source': 'AI_textbook', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [9]:
# Chroma 벡터저장소에 데이터(Document 객체)를 추가한다.
# from_documents() 메소드는 벡터저장소를 만듬과 동시에 데이터가 저장되지만 기존 벡터저장소에 새로운 데이터를 추가하려면 add_documents() 메소드를 사용한다.
added_doc_ids = chroma_db.add_documents(
    documents=doc_objects, # 벡터저장소에 저장할 데이터를 지정한다. 저장할 데이터 타입은 Document 객체가 저장된 리스트 타입이어야 한다.
    ids=doc_ids
)

In [10]:
# add_documents() 메소드는 벡터저장소에 데이터를 추가하고 ids를 리턴한다. len() 함수를 사용해서 추가된 데이터의 개수를 얻어올 수 있다.
print(len(added_doc_ids))

5


In [11]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook', 'chapter': 'Chapter 1'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 2'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 3'},
  {'chapter': 'Chapter 4', 'source': 'AI_textbook'},
  {'chapter': 'Chapter 5', 'source': 'AI_textbook'}]}

## 유사도 검사를 이용한 문서 검색

similarity_search() 메소드는 벡터저장소를 검색기로 만들지 않은 상태에서 유사도 검색을 해서 유사도가 높은 순서대로 지정한 개수 만큼 얻어온다.

In [12]:
query = '인공지능과 머신러닝의 관계는?'
# chroma 벡터저장소에서 유사도 검색을 한다.
results = chroma_db.similarity_search(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook, Chapter 2]
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook, Chapter 3]


## 문서 수정

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 내용을 덮어씌워서 수정한다.

In [13]:
# 수정할 새로운 문서 객체를 생성한다.
update_document1 = Document(
    page_content = '인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 11'}
)

update_document2 = Document(
    page_content = '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 22'}
)

update_document3 = Document(
    page_content = '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 33'}
)

In [14]:
# 단일 문서 수정
# update_document() 메소드로 수정할 ID 한 개와 수정할 내용을 지정해서 해당 데이터를 교체한다.
chroma_db.update_document(document_id='DOC_1', document=update_document1 )

In [15]:
# 여러 문서 일괄 수정
# update_documents() 메소드로 수정할 ID 여러 개와 수정할 내용을 지정해서 해당 데이터를 교체한다. 리스트로 묶어서 넘겨야 한다.
chroma_db.update_documents(ids=['DOC_2', 'DOC_3'], documents=[update_document2, update_document3])

In [16]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
  '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 'Chapter 11', 'source': 'AI_textbook'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 22'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 33'},
  {'chapter': 'Chapter 4', 'source': 'AI_textbook'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 5'}]}

In [17]:
query = '인공지능과 머신러닝의 관계는?'
results = chroma_db.similarity_search(query, k=2)
print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다. [출처: AI_textbook, Chapter 22]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]


## 문서 삭제

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 삭제한다.

In [18]:
# delete() 메소드로 삭제할 ID 한 개를 지정하면 해당 ID의 문서 한 개가 삭제된다.
chroma_db.delete(ids='DOC_1')

In [20]:
# delete() 메소드로 삭제할 ID 두 개 이상을 리스트로 묶어서 지정하면 해당 ID의 문서 여러 개가 삭제된다.
chroma_db.delete(ids=['DOC_2', 'DOC_3'])

In [26]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 'Chapter 1', 'source': 'AI_textbook'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 2'},
  {'chapter': 'Chapter 3', 'source': 'AI_textbook'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 4'},
  {'chapter': 'Chapter 5', 'source': 'AI_textbook'}]}

In [22]:
# delete_collection() 메소드는 테이블 자체를 제거한다. 따라서, 그 안에 저장된 모든 데이터도 삭제된다.
# delete_collection() 메소드 실행 후 get() 메소드를 실행하면 delete_collection() 메소드에 의해서 내용을 확인할 테이블 자체가 삭제되기 때문에 에러가 발생된다.
chroma_db.delete_collection()

## 문서 검색

In [25]:
chroma_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

added_doc_ids = chroma_db.add_documents(
    documents=doc_objects,
    ids=doc_ids
)